# 00 — One-Shot Fully-Automated Calibration

**The fastest path from raw image to a usable detector calibration.**

If you have:
* a calibrant image (CeO₂, LaB₆, Si, Al₂O₃, …)
* the X-ray wavelength
* the pixel pitch

… that is enough for the v2 pipeline to:

1. **Seed** the beam centre and sample–detector distance from the rings (no manual guess).
2. **Refine** all 5 geometry parameters and the 15-coefficient harmonic distortion via the differentiable LM engine.
3. **Build the empirical residual-correction map** that closes the last ~50–100 µε gap vs. the v1 C reference.
4. **Save** a v1-compatible `residual_corr.bin` and a JSON summary, both directly consumable by `midas_integrate`, `midas_integrate_v2`, and `CalibrantIntegratorOMP`.

All of that is one function call:

```python
result = calibrate(image, wavelength=..., pxY=..., dark=..., calibrant=...)
```

End-to-end runtime on a 2880² Perkin-Elmer image is ~5 minutes on CPU and a few seconds for the seed stage.

## 1. Environment + load a calibrant image

Any 2-D numpy array works.  This notebook reads an HDF5 file, but `.tif` / `.zarr` / GE-binary are all fine — `calibrate()` accepts an array.

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
import h5py, numpy as np

# Point this at a calibrant image you have.  The example file used here
# is a 2-second CeO2 exposure on a Perkin-Elmer 2880x2880 @ 150 um at 1ID.
DATA = Path(os.environ.get(
    'V2_ONESHOT_FILE',
    '/Users/hsharma/Desktop/analysis/leighanne/data/CeO2_XRD_echem_JLi_002587.vrx.h5',
))
assert DATA.exists(), f'set V2_ONESHOT_FILE or update the DATA path; missing {DATA}'

with h5py.File(DATA, 'r') as f:
    img  = f['exchange/data'][0].astype(np.float32)
    dark = f['exchange/data_dark'][0].astype(np.float32)
print(f'image  : shape={img.shape}  dtype={img.dtype}  mean={img.mean():.1f}')
print(f'dark   : shape={dark.shape}  mean={dark.mean():.1f}')

## 2. One function call — `calibrate(...)` does everything

Required inputs:

| arg | meaning |
|---|---|
| `image` | 2-D array, MIDAS convention `(NrPixelsZ, NrPixelsY)` |
| `wavelength` | Å (e.g. 0.184139 for 67.332 keV) |
| `pxY` | µm; if pixels aren't square also pass `pxZ` |

Optional:

| arg | default | what it does |
|---|---|---|
| `dark` | None | subtracted from `image` |
| `calibrant` | `"CeO2"` | `"CeO2"` / `"LaB6"` / `"Si"` / `"Al2O3"`, or a dict `{"a":..., "sg":...}` |
| `output_dir` | None | writes `calibration.json` + `residual_corr.bin` |
| `build_residual_corr` | True | port of v1 C `dg_residual_corr_lookup` |
| `device` | `"cpu"` | `"cuda"` for the LM step (seed stays on CPU) |

The internal `RhoD` is sanity-checked against your detector dimensions — pass a wrong-unit value (pixels instead of µm) and you'll get a clear `ValueError` instead of silently broken distortion.

In [ ]:
from midas_calibrate_v2 import calibrate

result = calibrate(
    img,
    wavelength=0.184139,           # Å — the only physics input
    pxY=150.0,                     # µm
    dark=dark,                     # optional but recommended
    calibrant='CeO2',              # name or {'a':..., 'sg':...} dict
    output_dir='./calib_out',      # writes residual_corr.bin + calibration.json
)

## 3. Inspect the result

The `AutoCalibrationResult` object carries the full refined state plus all the provenance you need to reproduce, plot, or debug.

In [ ]:
print(f'GEOMETRY (refined)')
print(f'  Lsd    = {result.Lsd:.3f} \u00b5m  ({result.Lsd/1000:.3f} mm)')
print(f'  BC     = ({result.BC_y:.4f}, {result.BC_z:.4f})')
print(f'  tilts  = tx={result.tx:+.4f}\u00b0  ty={result.ty:+.4f}\u00b0  tz={result.tz:+.4f}\u00b0')
print(f'  px     = ({result.pxY}, {result.pxZ}) \u00b5m')
print(f'  detector size = {result.NrPixelsY}\u00d7{result.NrPixelsZ}')
print()
print(f'QUALITY')
print(f'  in-loop strain (last LM iter) : {result.in_loop_strain_uE:.1f} \u00b5\u03b5')
print(f'  post-residual strain           : {result.post_residual_strain_uE:.1f} \u00b5\u03b5')
print()
print(f'SEED PROVENANCE  (auto BC/Lsd before LM)')
print(f'  seed BC = ({result.seed_BC_y:.2f}, {result.seed_BC_z:.2f})')
print(f'  seed Lsd = {result.seed_Lsd/1000:.3f} mm')
print(f'  seed time: {result.seed_seconds:.1f} s   refine time: {result.refine_seconds:.1f} s')
print()
print(f'DISTORTION (refined harmonics)')
for name, val in list(result.distortion.items())[:8]:
    print(f'  {name:8s} = {val:+.4e}')
print()
print(f'OUTPUTS')
print(f'  residual map: shape={tuple(result.residual_corr_map.shape)} dtype={result.residual_corr_map.dtype}')
print(f'  binary file : {result.residual_corr_bin_path}')

## 4. Visual sanity check — predicted rings should sit on the bright peaks

Always look at this.  Numbers can be wrong while passing all unit tests; the eye catches it instantly.

In [ ]:
from midas_hkls import SpaceGroup, Lattice, generate_hkls
import math, matplotlib.pyplot as plt

# Predict CeO₂ ring positions at the refined geometry using the same
# extinction rules midas_calibrate_v2 uses internally.
refs = generate_hkls(
    SpaceGroup.from_number(225),
    Lattice(a=5.4116, b=5.4116, c=5.4116, alpha=90.0, beta=90.0, gamma=90.0),
    wavelength_A=result.wavelength_A, two_theta_max_deg=14.0,
)
ring_R = [result.Lsd * math.tan(math.radians(r.two_theta_deg)) / result.pxY for r in refs]

img_d = np.clip(img - dark, 0, None)
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.log1p(img_d.clip(min=1)), origin='lower', cmap='gray',
          vmin=np.log1p(np.percentile(img_d[img_d>0], 60)),
          vmax=np.log1p(np.percentile(img_d[img_d>0], 99.95)))
th = np.linspace(0, 2*np.pi, 360)
for r in ring_R[:12]:
    ax.plot(result.BC_y + r*np.cos(th), result.BC_z + r*np.sin(th),
            '-', color='lime', lw=0.7, alpha=0.85)
ax.plot(result.BC_y, result.BC_z, 'ro', ms=8, mec='yellow', label='BC')
ax.set_title(f'CeO₂ image + predicted rings at refined geometry\n'
             f'Lsd={result.Lsd/1000:.2f} mm  BC=({result.BC_y:.1f}, {result.BC_z:.1f})  '
             f'strain={result.post_residual_strain_uE:.0f} µε')
ax.set_xlim(0, result.NrPixelsY); ax.set_ylim(0, result.NrPixelsZ)
ax.legend(); plt.tight_layout(); plt.show()


## 5. The residual-correction map

Empirical smooth $\Delta R(Y, Z)$ field that absorbs systematic deviations the harmonic distortion model can't capture — the v2 port of v1 C's `dg_residual_corr_lookup`. Differentiable via `torch.nn.functional.grid_sample`; storage is bit-identical to the binary v1 produces, so the same `.bin` file flows downstream into all three packages.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
cm = result.residual_corr_map.cpu().numpy()
vlim = float(np.percentile(np.abs(cm), 99))
im = ax.imshow(cm, origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
ax.plot(result.BC_y, result.BC_z, 'ko', ms=6)
ax.set_title(f'Residual correction map \u0394R(Y, Z) in pixels\n'
             f'rms = {float(np.sqrt((cm**2).mean())):.3f} px,  '
             f'range = [{cm.min():.3f}, {cm.max():.3f}] px')
plt.colorbar(im, ax=ax, label='\u0394R (px)')
plt.tight_layout(); plt.show()

## 6. Cross-package handoff

The `residual_corr.bin` is the same wire format as v1 C. Load it directly with `midas_integrate` or `midas_integrate_v2`:

In [ ]:
# Roundtrip check: load the binary back into both packages.
from midas_calibrate_v2.forward.residual_corr import load_residual_corr_bin
v2_load = load_residual_corr_bin(result.residual_corr_bin_path,
                                  NrPixelsY=result.NrPixelsY,
                                  NrPixelsZ=result.NrPixelsZ)
print('midas_calibrate_v2 roundtrip max diff:',
      float((v2_load - result.residual_corr_map).abs().max()))

try:
    from midas_integrate.residual_corr import load_residual_correction_map
    v1 = load_residual_correction_map(result.residual_corr_bin_path,
                                       NrPixelsY=result.NrPixelsY,
                                       NrPixelsZ=result.NrPixelsZ)
    print('midas_integrate roundtrip max diff   :',
          float(np.max(np.abs(v1.map - result.residual_corr_map.cpu().numpy()))))
except ImportError:
    print('midas_integrate not installed — skipped')

## 7. When the one-shot path isn't enough

* Multi-distance setup (sample moved between two exposures): use **NB 07 — multi-distance**.
* Want σ on every parameter (Bayesian uncertainty): **NB 02 / NB 11**.
* Multi-panel detectors (Pilatus 6M): **NB 03**.
* Non-cubic calibrant (corundum etc.): pass a `dict` with `a`, `c`, `alpha`, `gamma`, `sg`.
* First-time calibration with no good initial guess: **NB 06**.
* Refining only a subset (e.g. Lsd locked, only BC + tilts free): use `autocalibrate` directly with a custom `Refine` dict.